In [ ]:
import polars as pl
from tcrtrifold.tcrdock_utils import (
    dgeom_ndarr_from_dgeom_series,
    mn_distr_from_dgeom_ndarr,
    mn_distance_from,
    un_cossin_embed,
)
from tcrtrifold.utils import filter_to_cog_thresh, FORMAT_ANTIGEN_COLS, FORMAT_TCR_COLS


iedb_II_conf_old = pl.read_parquet(
    "../../data/iedb_II/triad/staged/iedb_II_triad.conf_af3.parquet"
)
iedb_II_tcrdock = pl.read_parquet(
    "../../data/iedb_II/triad/staged/iedb_II_triad.af3_tcrdock.parquet"
)

iedb_II_old = iedb_II_conf_old.join(
    iedb_II_tcrdock.select("pred_dgeom_4", "job_name").with_columns(
        pl.col("pred_dgeom_4").struct.unnest()
    ),
    on="job_name",
    how="inner",
)

template_dgeom = pl.read_csv(
    "../../data/pdb/raw/ternary_templates_v2.tsv", separator="\t"
).with_columns(
    pl.struct(
        **{
            k: pl.col(k)
            for k in [
                "d",
                "torsion",
                "mhc_unit_x_is_negative",
                "tcr_unit_y",
                "tcr_unit_z",
                "tcr_unit_x_is_negative",
                "mhc_unit_y",
                "mhc_unit_z",
            ]
        }
    ).alias("dgeom"),
    pl.when(pl.col("mhc_class") == 1)
    .then(pl.lit("I"))
    .otherwise(pl.lit("II"))
    .alias("mhc_class"),
)

class_II_t_dgeom = dgeom_ndarr_from_dgeom_series(
    template_dgeom.filter(pl.col("mhc_class") == "II").select("dgeom").to_series()
)

class_II_distr = mn_distr_from_dgeom_ndarr(class_II_t_dgeom)

_, iedb_II_p_dgeom = mn_distance_from(
    dgeom_ndarr_from_dgeom_series(iedb_II_old.select("pred_dgeom_4").to_series()),
    *class_II_distr,
)

iedb_II_old = iedb_II_old.with_columns(
    pl.Series(name="p_dgeom", values=iedb_II_p_dgeom)
)

iedb_II_old = iedb_II_old.explode("references")

iedb_II_conf_new = pl.read_parquet(
    "../../data/iedb_II/triad/iedb_II_triad.conf_af3.parquet"
)
iedb_II_new = iedb_II_conf_new.join(
    iedb_II_tcrdock.select("pred_dgeom_4", "job_name").with_columns(
        pl.col("pred_dgeom_4").struct.unnest()
    ),
    on="job_name",
    how="inner",
)
_, iedb_II_p_dgeom = mn_distance_from(
    dgeom_ndarr_from_dgeom_series(iedb_II_new.select("pred_dgeom_4").to_series()),
    *class_II_distr,
)

iedb_II_new = iedb_II_new.with_columns(
    pl.Series(name="p_dgeom", values=iedb_II_p_dgeom)
)

In [23]:
from tcrtrifold.utils import FORMAT_ANTIGEN_COLS
from tcrtrifold.eval_utils import antigen_raw_score_auc
import sklearn.metrics as metrics
from scipy.stats import mannwhitneyu, norm, false_discovery_control
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import OrderedDict, defaultdict


def auc_by(
    df, featnames, feat_type, grouping_cols=FORMAT_ANTIGEN_COLS + ["references"]
):

    antigen_st = df.filter(pl.col("cognate")).select(grouping_cols).unique()

    feat_df = []

    for row in antigen_st.iter_rows(named=True):

        focal_a_st = pl.DataFrame([row]).select(pl.exclude("job_name"))
        focal_pos = df.join(focal_a_st, on=grouping_cols)
        focal_neg = df.join(
            focal_pos.select(FORMAT_ANTIGEN_COLS).unique(),
            on=FORMAT_ANTIGEN_COLS,
        ).filter(~pl.col("cognate"))

        focal_triad = pl.concat([focal_pos, focal_neg])

        for feat, ft in zip(featnames, feat_type):
            dat = focal_triad.select(feat, "cognate").to_numpy()

            fpr, tpr, threshold = metrics.roc_curve(dat[:, 1], dat[:, 0])
            # roc_auc = abs(metrics.auc(fpr, tpr) - 0.5) + 0.5
            roc_auc = metrics.auc(fpr, tpr)

            feat_df.append(
                {
                    "auc": roc_auc,
                    # "fpr": list(fpr),
                    # "tpr": list(tpr),
                    "featname": feat,
                    "feat_type": ft,
                }
            )

    feat_df = pl.DataFrame(feat_df)

    return feat_df


docking_feats = [
    "d",
    "mhc_unit_y",
    "mhc_unit_z",
    "tcr_unit_y",
    "tcr_unit_z",
    "torsion",
    "p_dgeom",
]

interface_feats = [
    "mean_p_tcr_pae",
    "mean_tcr_p_pae",
    "mean_mhc_tcr_pae",
    "mean_tcr_mhc_pae",
    # "mean_p_tcr_contact_prob",
    # "mean_tcr_p_contact_prob",
    # "mean_mhc_tcr_contact_prob",
    # "mean_tcr_mhc_contact_prob",
    # "mean_p_tcr_interface_pae",
    # "mean_tcr_p_interface_pae",
    # "mean_tcr_pmhc_interface_pae",
    # "mean_pmhc_tcr_interface_pae",
    "mean_p_tcr_interface_contact_prob",
    # "mean_tcr_p_interface_contact_prob",
    # "mean_tcr_pmhc_interface_contact_prob",
    # "mean_pmhc_tcr_interface_contact_prob",
    "mean_p_mhc_pae",
    # "mean_mhc_p_pae",
    # "mean_mhc_p_interface_pae",
    # "mean_p_mhc_interface_pae",
    # "mean_mhc_p_contact_prob",
    # "mean_p_mhc_contact_prob",
    # "min_p_tcr_pae",
    # "min_mhc_tcr_pae",
    # "min_tcr_p_pae",
    # "min_tcr_mhc_pae",
    "tcr_mhc_contacts",
    # "tcr_p_contacts",
]


local_feats_II = [
    "peptide_mean_pLDDT",
    "peptide_mean_pLDDT_II",
    "tcr_1_cdr_1_mean_pLDDT",
    "tcr_1_cdr_2_mean_pLDDT",
    "tcr_1_cdr_2_5_mean_pLDDT",
    "tcr_1_cdr_3_mean_pLDDT",
    "tcr_2_cdr_1_mean_pLDDT",
    "tcr_2_cdr_2_mean_pLDDT",
    "tcr_2_cdr_2_5_mean_pLDDT",
    "tcr_2_cdr_3_mean_pLDDT",
    "tcr_cdrs_mean_pLDDT",
    "mhc_helices_mean_pLDDT",
]


local_feats_I = [
    "peptide_mean_pLDDT",
    "tcr_1_cdr_1_mean_pLDDT",
    "tcr_1_cdr_2_mean_pLDDT",
    "tcr_1_cdr_2_5_mean_pLDDT",
    "tcr_1_cdr_3_mean_pLDDT",
    "tcr_2_cdr_1_mean_pLDDT",
    "tcr_2_cdr_2_mean_pLDDT",
    "tcr_2_cdr_2_5_mean_pLDDT",
    "tcr_2_cdr_3_mean_pLDDT",
    "tcr_cdrs_mean_pLDDT",
    "mhc_helices_mean_pLDDT",
]

summary_feats = [
    "iptm",
    "ptm",
    "ranking_score",
]

featnames_II = docking_feats + interface_feats + local_feats_II + summary_feats
feat_type_II = (
    ["docking"] * len(docking_feats)
    + ["interface"] * len(interface_feats)
    + ["local"] * len(local_feats_II)
    + ["summary"] * len(summary_feats)
)
featnames_I = interface_feats + local_feats_I + summary_feats
feat_type_I = (
    # ["docking"] * len(docking_feats)
    ["interface"] * len(interface_feats)
    + ["local"] * len(local_feats_I)
    + ["summary"] * len(summary_feats)
)

In [24]:
pl.Config.set_tbl_rows(-1)

df_unweighted = auc_by(
    iedb_II_old, featnames_II, feat_type_II, grouping_cols=FORMAT_ANTIGEN_COLS
)

df_unweighted_agg = (
    df_unweighted.group_by("featname", "feat_type")
    .agg(pl.col("auc").median().alias("mean_auc"), pl.col("auc"))
    .sort(by="mean_auc", descending=True)
)

df_unweighted_agg

featname,feat_type,mean_auc,auc
str,str,f64,list[f64]
"""mhc_helices_mean_pLDDT""","""local""",0.76,"[0.713889, 0.96, … 0.8]"
"""peptide_mean_pLDDT""","""local""",0.737263,"[0.729861, 1.0, … 0.76]"
"""tcr_mhc_contacts""","""interface""",0.73075,"[0.774306, 0.975, … 0.765]"
"""ptm""","""summary""",0.726307,"[0.698264, 0.995, … 0.805]"
"""ranking_score""","""summary""",0.725,"[0.693403, 1.0, … 0.865]"
"""iptm""","""summary""",0.721405,"[0.696875, 1.0, … 0.8]"
"""tcr_2_cdr_2_mean_pLDDT""","""local""",0.7,"[0.70625, 0.98, … 0.67]"
"""tcr_2_cdr_1_mean_pLDDT""","""local""",0.68,"[0.709028, 0.695, … 0.5]"
"""tcr_cdrs_mean_pLDDT""","""local""",0.676531,"[0.697917, 0.895, … 0.85]"


In [26]:
pl.Config.set_tbl_rows(-1)

df_unweighted = auc_by(
    iedb_II_new, featnames_II, feat_type_II, grouping_cols=FORMAT_ANTIGEN_COLS
)

df_unweighted_agg = (
    df_unweighted.group_by("featname", "feat_type")
    .agg(pl.col("auc").median().alias("median_auc"), pl.col("auc"))
    .sort(by="median_auc", descending=True)
)

df_unweighted_agg

featname,feat_type,median_auc,auc
str,str,f64,list[f64]
"""peptide_mean_pLDDT""","""local""",0.74,"[0.56, 1.0, … 0.72]"
"""peptide_mean_pLDDT_II""","""local""",0.74,"[0.57, 1.0, … 0.7]"
"""mhc_helices_mean_pLDDT""","""local""",0.735,"[0.75, 1.0, … 0.24]"
"""ptm""","""summary""",0.726307,"[0.71, 1.0, … 0.24]"
"""ranking_score""","""summary""",0.725,"[0.735, 1.0, … 0.245]"
"""iptm""","""summary""",0.721405,"[0.695, 1.0, … 0.245]"
"""tcr_mhc_contacts""","""interface""",0.72,"[0.645, 0.96, … 0.255]"
"""tcr_2_cdr_2_mean_pLDDT""","""local""",0.7,"[0.83, 1.0, … 0.26]"
"""tcr_2_cdr_1_mean_pLDDT""","""local""",0.68,"[0.65, 1.0, … 0.36]"


In [27]:
pl.Config.set_tbl_rows(-1)

df_weighted = auc_by(
    iedb_II_new, featnames_II, feat_type_II,
)

df_weighted_agg = (
    df_weighted.group_by("featname", "feat_type")
    .agg(pl.col("auc").median().alias("median_auc"), pl.col("auc"))
    .sort(by="median_auc", descending=True)
)

df_weighted_agg

featname,feat_type,median_auc,auc
str,str,f64,list[f64]
"""peptide_mean_pLDDT""","""local""",0.74,"[0.07, 0.67, … 0.15]"
"""peptide_mean_pLDDT_II""","""local""",0.74,"[0.11, 0.69, … 0.14]"
"""mhc_helices_mean_pLDDT""","""local""",0.739281,"[0.0, 0.74, … 0.15]"
"""ranking_score""","""summary""",0.7325,"[0.105, 0.815, … 0.245]"
"""tcr_mhc_contacts""","""interface""",0.73075,"[0.345, 0.94, … 0.33]"
"""ptm""","""summary""",0.728154,"[0.115, 0.805, … 0.245]"
"""iptm""","""summary""",0.728,"[0.1, 0.8, … 0.235]"
"""tcr_2_cdr_2_mean_pLDDT""","""local""",0.7,"[0.17, 0.68, … 0.3]"
"""tcr_cdrs_mean_pLDDT""","""local""",0.69,"[0.08, 0.76, … 0.24]"
